#Kelompok 11
**Nama Anggota Kelompok**

1. Athaalla Rayya Genaro I - 5026221116
2. Raihan Fareliansyah - 5026221160
3. Rayhan Lauzzadani - 5026221186

#Topics
Optimasi Rute Inspeksi Kebersihan Taman Di Surabaya (Inspektur : Dinas Lingkungan Hidup)

# **Install Library**

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import folium
from folium.plugins import PolyLineTextPath
from geopy.distance import geodesic
from datetime import timedelta, datetime
from sklearn.cluster import KMeans
from tabulate import tabulate

#**Import Kaggle**

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR=
# NOTEBOOK.

!pip install -q kagglehub
!pip install tabulate

import kagglehub
rayhanlauzzadani_datasetfp_path = kagglehub.dataset_download('rayhanlauzzadani/datasetsc-fp')
print(f" Dataset downloaded to: {rayhanlauzzadani_datasetfp_path}")

100%|██████████| 1.67k/1.67k [00:00<00:00, 1.97MB/s]

Extracting files...
 Dataset downloaded to: /root/.cache/kagglehub/datasets/rayhanlauzzadani/datasetsc-fp/versions/1


# **Load Dataset**

In [ ]:
# Ganti path ke file baru
df = pd.read_csv(f"{rayhanlauzzadani_datasetfp_path}/dataset_taman_surabaya_fixed.csv")

# Ambil koordinat dari baris yang mengandung "Dinas Lingkungan Hidup"
dlh_row = df[df['Nama Taman'].str.contains("Dinas Lingkungan Hidup", case=False, na=False)].iloc[0]
dlh_coord = (dlh_row['Latitude'], dlh_row['Longitude'])

# Tampilkan dataframe
print(tabulate(df, headers='keys', tablefmt='psql'))

+----+-------------------------------------------+------------+-------------+----------------+
|    | Nama Taman                                |   Latitude |   Longitude | Kategori       |
|----+-------------------------------------------+------------+-------------+----------------|
|  0 | Kantor Dinas Lingkungan Hidup             |   -7.27839 |     112.763 | pemerintah     |
|  1 | Taman. Lansia                             |   -7.271   |     112.75  | pemerintah     |
|  2 | Taman Flora                               |   -7.29427 |     112.762 | pemerintah     |
|  3 | Taman. Persahabatan                       |   -7.27673 |     112.746 | pemerintah     |
|  4 | Taman BMX Ketabang                        |   -7.26368 |     112.75  | pemerintah     |
|  5 | Taman Keputran                            |   -7.27319 |     112.744 | pemerintah     |
|  6 | Taman Ngagel                              |   -7.28844 |     112.745 | pemerintah     |
|  7 | Taman AIS Nasution                        |

# **Clustering Menjadi 3 Grup**

In [ ]:
# KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(df[['Latitude', 'Longitude']])

# Output: jumlah taman per cluster
print("Jumlah taman per cluster:")
print(df['Cluster'].value_counts().sort_index())

# Output: total taman (tanpa DLH)
total_taman_tanpa_dlh = len(df[df['Nama Taman'] != dlh_row['Nama Taman']])
print("Total taman (tanpa DLH):", total_taman_tanpa_dlh)

Jumlah taman per cluster:
Cluster
0    18
1    25
2     8
Name: count, dtype: int64
Total taman (tanpa DLH): 50


# **GA-VRP**

## **Distance Matrix Construction for GA-VRP**

In [ ]:
from geopy.distance import geodesic
import numpy as np

def create_distance_matrix(coords):
    n = len(coords)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = geodesic(coords[i], coords[j]).kilometers
    return matrix

# Pilih cluster, misal cluster 0 (ganti ke 1/2 jika perlu)
cluster_df = df[df['Cluster'] == 0].reset_index(drop=True)
coords = cluster_df[['Latitude', 'Longitude']].values
dist_matrix = create_distance_matrix(coords)

## **Implementasi Fungsi GA untuk Vehicle Routing Problem**

In [ ]:
def ga_vrp_tw_multitrip(
    cluster_df, n_vehicle, speed_kmh=30,
    pop_size=70, n_generations=170, mutation_rate=0.10,
    tournament_size=10, elitism_count=10, max_work_time=480):

    coords = cluster_df[['Latitude', 'Longitude']].values
    n = len(coords)
    service_times = cluster_df['service_time'].tolist()

    # 1. Hitung jarak antar node hanya sekali
    dist_matrix = create_distance_matrix(coords)

    # 2. Generate populasi spatial-aware, radial seperti “pizza slice”
    def generate_population(pop_size, n_cities):
        pop = []
        depot = coords[0]
        nodes_df = cluster_df.iloc[1:].copy()
        nodes_df['angle'] = np.arctan2(
            nodes_df['Latitude']-depot[0],
            nodes_df['Longitude']-depot[1]
        )
        nodes_df = nodes_df.sort_values('angle')
        sorted_idx = list(nodes_df.index)

        sector_size = len(sorted_idx) // n_vehicle
        for _ in range(pop_size):
            routes = []
            assigned = 0
            for v in range(n_vehicle):
                if v == n_vehicle-1:
                    chunk = sorted_idx[assigned:]
                else:
                    chunk = sorted_idx[assigned:assigned+sector_size]
                assigned += len(chunk)
                if len(chunk) > 2:
                    i = random.randint(0, len(chunk)-2)
                    chunk[i], chunk[i+1] = chunk[i+1], chunk[i]
                routes.append([0] + list(chunk) + [0] if chunk else [0,0])
            pop.append(routes)
        return pop

    def has_crossing(route):
        def ccw(A,B,C):
            return (C[1]-A[1])*(B[0]-A[0]) > (B[1]-A[1])*(C[0]-A[0])
        nodes = route
        pts = coords[nodes]
        for i in range(1,len(pts)-2):
            for j in range(i+1, len(pts)-1):
                A,B = pts[i-1], pts[i]
                C,D = pts[j-1], pts[j]
                if abs(i-j)<=1: continue
                if ccw(A,C,D)!=ccw(B,C,D) and ccw(A,B,C)!=ccw(A,B,D):
                    return True
        return False

    def repair(routes):
        allnodes = [i for r in routes for i in r[1:-1]]
        count = {i:allnodes.count(i) for i in range(1,n)}
        for v in range(len(routes)):
            clean = []
            for node in routes[v][1:-1]:
                if count[node]>1:
                    count[node]-=1
                else:
                    clean.append(node)
            routes[v] = [0]+clean+[0]
        missing = [i for i in range(1,n) if count[i]==0]
        for m in missing:
            idx = np.argmin([len(r) for r in routes])
            routes[idx].insert(-1, m)
        return routes

    def mutate(routes):
        routes = [r[:] for r in routes]
        for v in range(n_vehicle):
            if len(routes[v])>3 and random.random()<mutation_rate:
                idx = range(1, len(routes[v])-1)
                i = random.choice(idx)
                if i < len(routes[v])-2:
                    routes[v][i], routes[v][i+1] = routes[v][i+1], routes[v][i]
        if n_vehicle>1 and random.random()<mutation_rate:
            v1, v2 = random.sample(range(n_vehicle), 2)
            if len(routes[v1])>2 and len(routes[v2])>2:
                routes[v1][1], routes[v2][1] = routes[v2][1], routes[v1][1]
        return repair(routes)

    def crossover(p1, p2):
        child = []
        for v in range(n_vehicle):
            blok1 = p1[v][1:-1]
            blok2 = p2[v][1:-1]
            chosen = blok1 if random.random()<0.5 else blok2
            child.append([0]+chosen+[0] if chosen else [0,0])
        return repair(child)

    def fitness(routes):
        total = 0
        penalty = 0
        seen = set()
        for v, route in enumerate(routes):
            t = 0
            for i in range(len(route)-1):
                t += dist_matrix[route[i], route[i+1]] / speed_kmh * 60
                if i < len(route)-2:
                    t += service_times[route[i+1]]
            if t > max_work_time: penalty += 1e5 * (t - max_work_time)
            total += sum([dist_matrix[route[i], route[i+1]] for i in range(len(route)-1)])
            if has_crossing(route): penalty += 1e7
        allnodes = [i for r in routes for i in r[1:-1]]
        if len(set(allnodes)) < n-1: penalty += 1e5 * ((n-1)-len(set(allnodes)))
        return total + penalty

    # --- GA Main Loop ---
    population = generate_population(pop_size, n)
    best_routes = None
    best_score = float('inf')
    ga_fitness_history = []

    for gen in range(n_generations):
        population = [repair(r) for r in population]
        population = sorted(population, key=fitness)
        elites = population[:elitism_count]
        new_population = elites.copy()
        while len(new_population) < pop_size:
            p1, p2 = random.sample(population[:tournament_size], 2)
            child = crossover(p1, p2)
            child = mutate(child)
            new_population.append(child)
        population = new_population
        for routes in population:
            score = fitness(routes)
            if score < best_score:
                best_score = score
                best_routes = routes

    return dist_matrix, best_routes, cluster_df, ga_fitness_history

## **Proses VRP Setiap Cluster (DLH disisipkan di awal)**

In [ ]:
SEED = 39
random.seed(SEED)
np.random.seed(SEED)
VEHICLE_SPEED_KMH = 30
OPEN_TIME = 8 * 60
CLOSE_TIME = 16 * 60
MAX_WORK_TIME = 8 * 60
vehicle_per_cluster = {0: 2, 1: 3, 2: 1}

best_total_distance = float('inf')
best_seed = None
best_multitrip_final = None
for seed in range(1, 2):
    random.seed(SEED)
    np.random.seed(SEED)
    cluster_multitrip_results = []
    for cluster_id in sorted(df['Cluster'].unique()):
        n_vehicle = vehicle_per_cluster[cluster_id]
        # Ambil cluster tanpa depot DLH
        cluster_df = df[
            (df['Cluster'] == cluster_id) &
            (~df['Nama Taman'].str.contains("Dinas Lingkungan Hidup", case=False, na=False))
        ].copy().reset_index(drop=True)
        # Ambil depot DLH dan set Cluster sama
        dlh_row = df[df['Nama Taman'].str.contains("Dinas Lingkungan Hidup", case=False, na=False)].iloc[0]
        dlh_df = pd.DataFrame([dlh_row])
        dlh_df['Cluster'] = cluster_id
        cluster_df = pd.concat([dlh_df, cluster_df], ignore_index=True)

        if 'service_time' not in cluster_df.columns:
            def get_service_time(x):
                return 30 if str(x).lower().strip() == 'pemerintah' else 15
            cluster_df['service_time'] = cluster_df['Kategori'].apply(get_service_time) if 'Kategori' in cluster_df.columns else 30
        dist_matrix, trips, df_ref = ga_vrp_tw_multitrip(
            cluster_df, n_vehicle=n_vehicle, speed_kmh=VEHICLE_SPEED_KMH, max_work_time=MAX_WORK_TIME
        )
        for i, trip in enumerate(trips):
            trip_df = df_ref.iloc[trip].reset_index(drop=True)
            trip_df['Urutan'] = range(1, len(trip_df) + 1)
            trip_df['Cluster'] = cluster_id
            trip_df['Vehicle'] = i + 1
            cluster_multitrip_results.append(trip_df)
    df_multitrip_final_temp = pd.concat(cluster_multitrip_results, ignore_index=True)
    total_jarak = 0
    for cluster_id in sorted(df_multitrip_final_temp['Cluster'].unique()):
        for vehicle in sorted(df_multitrip_final_temp[df_multitrip_final_temp['Cluster']==cluster_id]['Vehicle'].unique()):
            r = df_multitrip_final_temp[
                (df_multitrip_final_temp['Cluster']==cluster_id)&
                (df_multitrip_final_temp['Vehicle']==vehicle)
            ].sort_values('Urutan')
            for i in range(len(r)-1):
                coord_a = (r.iloc[i]['Latitude'], r.iloc[i]['Longitude'])
                coord_b = (r.iloc[i+1]['Latitude'], r.iloc[i+1]['Longitude'])
                total_jarak += geodesic(coord_a, coord_b).kilometers
    if total_jarak < best_total_distance:
        best_total_distance = total_jarak
        best_seed = seed
        best_multitrip_final = df_multitrip_final_temp.copy()
df_multitrip_final = best_multitrip_final.copy()

ValueError: too many values to unpack (expected 3)

## **Visualisasi Pembagian Cluster**

In [ ]:
plt.figure(figsize=(10,8))
colors = ['red', 'blue', 'green']
for cluster_id, color in zip(sorted(df['Cluster'].unique()), colors):
    cluster_points = df[df['Cluster'] == cluster_id]
    plt.scatter(cluster_points['Longitude'], cluster_points['Latitude'], label=f'Cluster {cluster_id}', color=color, s=60)
plt.scatter(dlh_coord[1], dlh_coord[0], c='black', marker='X', s=200, label='DLH (Depot)')
plt.title('Pembagian Cluster Taman (Hasil KMeans)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

## **Visualisasi Rute VRP Optimal + DLH**

In [ ]:
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
           '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
           '#a60000', '#adff2f', '#009999', '#ff1493', '#654321']  # tambah jika rute > 10
markers = ['o', 's', '^', 'd', 'v', '*', 'P', 'X', '<', '>', 'h', 'p']

vehicle_ids = df_multitrip_final[['Cluster', 'Vehicle']].drop_duplicates().sort_values(['Cluster', 'Vehicle']).reset_index(drop=True)
vehicle_ids['color'] = [palette[i % len(palette)] for i in range(len(vehicle_ids))]
vehicle_ids['marker'] = [markers[i % len(markers)] for i in range(len(vehicle_ids))]
vehicle_color_map = {(row['Cluster'], row['Vehicle']): (row['color'], row['marker']) for _, row in vehicle_ids.iterrows()}

In [ ]:
plt.figure(figsize=(10,8))
for (cluster_id, vehicle), (color, marker) in vehicle_color_map.items():
    vdata = df_multitrip_final[(df_multitrip_final['Cluster']==cluster_id) & (df_multitrip_final['Vehicle']==vehicle)].sort_values('Urutan')
    label = f'Cluster {cluster_id} Vehicle {vehicle}'
    plt.plot(vdata['Longitude'], vdata['Latitude'], linestyle='-', color=color, marker=marker, label=label)
plt.scatter(dlh_coord[1], dlh_coord[0], c='black', marker='X', s=200, label='DLH (Depot)')
plt.title('Rute VRP Multi-Trip (GA, Feasible Time)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.legend()
plt.show()

## **Visualisasi Folium**

In [ ]:
hex_to_folium = {
    '#1f77b4': 'blue',
    '#ff7f0e': 'orange',
    '#2ca02c': 'green',
    '#d62728': 'red',
    '#9467bd': 'purple',
    '#8c564b': 'darkred',
    '#e377c2': 'pink',
    '#7f7f7f': 'gray',
    '#bcbd22': 'lightgreen',
    '#17becf': 'cadetblue'
}

m = folium.Map(location=dlh_coord, zoom_start=13)
for (cluster_id, vehicle), (color, marker) in vehicle_color_map.items():
    vdata = df_multitrip_final[
        (df_multitrip_final['Cluster'] == cluster_id) &
        (df_multitrip_final['Vehicle'] == vehicle)
    ].sort_values('Urutan')
    points = list(zip(vdata['Latitude'], vdata['Longitude']))
    line = folium.PolyLine(points, color=color, weight=4, opacity=0.7, tooltip=f'Cluster {cluster_id} Vehicle {vehicle}')
    line.add_to(m)
    # Marker ikon pohon per node
    for i, row in vdata.iterrows():
        color_name = hex_to_folium.get(color, 'blue')
        folium.Marker(
            location=[row['Latitude'], row['Longitude']],
            popup=f"{row['Nama Taman']}<br>Cluster {cluster_id} Vehicle {vehicle}<br>Urutan {row['Urutan']}",
            icon=folium.Icon(color=color_name, icon="tree", prefix='fa')
        ).add_to(m)
folium.Marker(location=dlh_coord, popup="DLH (Depot)", icon=folium.Icon(color="black", icon="home")).add_to(m)
m

## **Detail Jarak Antar Taman per Cluster (Urutan Rute GA)**


In [ ]:
for cluster_id in sorted(df_multitrip_final['Cluster'].unique()):
    for vehicle in sorted(df_multitrip_final[df_multitrip_final['Cluster']==cluster_id]['Vehicle'].unique()):
        r = df_multitrip_final[(df_multitrip_final['Cluster']==cluster_id)&(df_multitrip_final['Vehicle']==vehicle)].sort_values('Urutan').reset_index(drop=True)
        print(f"\nCluster {cluster_id} - Vehicle {vehicle}:")
        rows = []
        for i in range(len(r)-1):
            asal = r.iloc[i]
            tujuan = r.iloc[i+1]
            jarak = geodesic((asal['Latitude'], asal['Longitude']), (tujuan['Latitude'], tujuan['Longitude'])).kilometers
            rows.append([asal['Nama Taman'], tujuan['Nama Taman'], f"{jarak:.2f} km"])
        print(tabulate(rows, headers=['Dari', 'Ke', 'Jarak'], tablefmt='psql'))

## **Total Jarak & Waktu Tempuh Truk per Cluster (Fixed Speed 30 km/jam)**

In [ ]:
kecepatan_truk_kmh = VEHICLE_SPEED_KMH
hasil_per_cluster = []
for cluster_id in sorted(df_multitrip_final['Cluster'].unique()):
    for vehicle in sorted(df_multitrip_final[df_multitrip_final['Cluster']==cluster_id]['Vehicle'].unique()):
        r = df_multitrip_final[(df_multitrip_final['Cluster']==cluster_id)&(df_multitrip_final['Vehicle']==vehicle)].sort_values('Urutan').reset_index(drop=True)
        total_jarak = 0
        total_service = 0
        for i in range(len(r)-1):
            coord_a = (r.iloc[i]['Latitude'], r.iloc[i]['Longitude'])
            coord_b = (r.iloc[i+1]['Latitude'], r.iloc[i+1]['Longitude'])
            total_jarak += geodesic(coord_a, coord_b).kilometers
            if i < len(r)-1 and i > 0:
                total_service += r.iloc[i]['service_time']/60.0  # dalam jam
        waktu_jalan_jam = total_jarak / kecepatan_truk_kmh
        waktu_total_jam = waktu_jalan_jam + total_service
        jam = int(waktu_total_jam)
        menit = int((waktu_total_jam - jam) * 60)
        waktu_tempuh_format = f"{jam} jam {menit} menit" if jam > 0 else f"{menit} menit"
        hasil_per_cluster.append({
            'Cluster': cluster_id,
            'Vehicle': vehicle,
            'Jumlah Taman': len(r)-2,
            'Total Jarak (km)': round(total_jarak, 2),
            'Waktu Tempuh': waktu_tempuh_format
        })
hasil_df = pd.DataFrame(hasil_per_cluster)
print(tabulate(hasil_df, headers='keys', tablefmt='psql', showindex=False))

# **ACO-VRP**

## **Distance Matrix Construction for ACO-VRP**

In [ ]:
from geopy.distance import geodesic
import numpy as np

def create_distance_matrix(coords):
    n = len(coords)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = geodesic(coords[i], coords[j]).kilometers
    return matrix

def total_distance(path, matrix):
    return sum(matrix[path[i]][path[i+1]] for i in range(len(path)-1))

## **Implementasi Fungsi ACO untuk Vehicle Routing Problem**

In [ ]:
import numpy as np
import pandas as pd
from geopy.distance import geodesic
from datetime import datetime, timedelta
import random

class ACO_VRP:
    def __init__(self, n_vehicle, speed=30, n_ants=10, n_iter=100, alpha=1, beta=3,
                 evaporation=0.5, Q=100, random_seed=None):
        self.n_vehicle = n_vehicle
        self.speed = speed
        self.n_ants = n_ants
        self.n_iter = n_iter
        self.alpha = alpha
        self.beta = beta
        self.evaporation = evaporation
        self.Q = Q
        self.random_seed = random_seed

    def _select_next_node(self, curr_idx, unvisited, pheromone, heuristic):
        probs = []
        for j in unvisited:
            tau = pheromone[curr_idx][j] ** self.alpha
            eta = heuristic[curr_idx][j] ** self.beta
            probs.append(tau * eta)
        probs = np.array(probs)
        probs = probs / probs.sum()
        return random.choices(list(unvisited), weights=probs)[0]

    def solve_batch(self, coords, start_node=0):
        rng = np.random.default_rng(self.random_seed)
        n = len(coords)
        pheromone = np.ones((n, n))
        heuristic = 1 / (np.array([[geodesic(a, b).km for b in coords] for a in coords]) + 1e-10)
        best_path, best_cost = None, float('inf')

        for _ in range(self.n_iter):
            all_paths = []
            for _ in range(self.n_ants):
                path = [start_node]
                unvisited = set(range(n)) - {start_node}
                curr = start_node
                while unvisited:
                    nxt = self._select_next_node(curr, unvisited, pheromone, heuristic)
                    path.append(nxt)
                    unvisited.remove(nxt)
                    curr = nxt
                path.append(start_node)  # kembali ke depot
                cost = sum(geodesic(coords[path[i]], coords[path[i+1]]).km for i in range(len(path)-1))
                all_paths.append((path, cost))
                if cost < best_cost:
                    best_cost = cost
                    best_path = path

            pheromone *= (1 - self.evaporation)
            for path, dist in all_paths:
                for i in range(len(path) - 1):
                    pheromone[path[i]][path[i + 1]] += self.Q / dist

        return best_path

## **Proses VRP Setiap Cluster (DLH disisipkan di awal)**

In [ ]:
KECEPATAN_TRUK_KMH = 30
JAM_KERJA_MAX = 8
START_TIME = "08:00"
vehicle_per_cluster = {0: 2, 1: 3, 2: 1}
RANDOM_SEED = 12
cluster_results = []

for cluster_id in sorted(df['Cluster'].unique()):
    n_vehicle = vehicle_per_cluster.get(cluster_id, 1)
    # Hilangkan depot dari cluster_df dulu!
    cluster_df = df[
        (df['Cluster'] == cluster_id) &
        (~df['Nama Taman'].str.contains("Dinas Lingkungan Hidup", case=False, na=False))
    ].copy().reset_index(drop=True)
    depot_row = df[df['Nama Taman'].str.contains("Dinas Lingkungan Hidup", case=False, na=False)].iloc[0]
    depot_coord = (depot_row['Latitude'], depot_row['Longitude'])

    # Urutkan berdasarkan sudut (angle) dari depot
    cluster_df['angle'] = cluster_df.apply(
        lambda row: np.arctan2(row['Latitude'] - depot_coord[0], row['Longitude'] - depot_coord[1]), axis=1)
    cluster_df = cluster_df.sort_values('angle').reset_index(drop=True)  # <--- reset index lagi!

    indices = list(range(len(cluster_df)))
    batch_indices_list = np.array_split(indices, n_vehicle)

    time_start = datetime.strptime(START_TIME, "%H:%M")
    vehicle_num = 1

    for i, batch_indices in enumerate(batch_indices_list):
        # Pastikan batch_indices sudah unique dan tidak overlap!
        batch_df = pd.concat([depot_row.to_frame().T, cluster_df.iloc[list(batch_indices)].drop(columns='angle')],
                             ignore_index=True)
        coords = batch_df[['Latitude', 'Longitude']].values

        aco = ACO_VRP(n_vehicle=1, speed=KECEPATAN_TRUK_KMH, random_seed=RANDOM_SEED + i)
        path = aco.solve_batch(coords, start_node=0)

        # Hitung waktu trip
        time_accum = 0
        last_idx = 0
        for j in path[1:]:
            travel_time = geodesic(coords[last_idx], coords[j]).km / KECEPATAN_TRUK_KMH
            if j == 0:
                waktu_inspeksi = 0
            else:
                kategori = batch_df.iloc[j]['Kategori'].strip().lower()
                waktu_inspeksi = 0.5 if kategori == 'pemerintah' else 0.25
            time_accum += travel_time + waktu_inspeksi
            last_idx = j

        # Simpan trip
        route_result = batch_df.iloc[path].copy().reset_index(drop=True)
        route_result['Urutan'] = range(1, len(route_result) + 1)
        route_result['Kendaraan'] = vehicle_num
        route_result['Cluster'] = cluster_id
        route_result['Waktu Mulai'] = time_start.strftime('%H:%M')
        route_result['Waktu Selesai'] = (time_start + timedelta(hours=time_accum)).strftime('%H:%M')
        route_result['Total Jam'] = round(time_accum, 2)
        cluster_results.append(route_result)

        if time_accum > JAM_KERJA_MAX:
            print(f"Cluster {cluster_id} - Kendaraan {vehicle_num} melebihi batas waktu: {time_accum:.2f} jam")

        time_start += timedelta(hours=time_accum)
        vehicle_num += 1

# Gabungkan hasil seluruh cluster
final_df = pd.concat(cluster_results, ignore_index=True)
df_rute_final = final_df.copy()

## **Visualisasi Pembagian Cluster**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
colors = ['green', 'red', 'blue', 'orange', 'purple', 'cyan']
for cluster_id, color in zip(sorted(df['Cluster'].unique()), colors):
    cluster_df = df[df['Cluster'] == cluster_id]
    plt.scatter(cluster_df['Longitude'], cluster_df['Latitude'], label=f'Cluster {cluster_id}', color=color, s=60)

plt.scatter(dlh_coord[1], dlh_coord[0], c='black', marker='X', s=200, label='DLH (Depot)')
plt.title('Sebaran Taman dan Cluster (Hasil KMeans)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

## **Visualisasi Rute VRP Optimal + DLH**

In [ ]:
import matplotlib.pyplot as plt
import itertools

plt.figure(figsize=(10, 8))
# Banyak warna, supaya cukup untuk semua kendaraan!
color_list = ['green', 'red', 'blue', 'orange', 'purple', 'cyan', 'brown', 'magenta', 'olive', 'grey']
color_cycle = itertools.cycle(color_list)

for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    cluster_rute = df_rute_final[df_rute_final['Cluster'] == cluster_id]
    for kendaraan in sorted(cluster_rute['Kendaraan'].unique()):
        trip = cluster_rute[cluster_rute['Kendaraan'] == kendaraan].sort_values('Urutan')
        trip = pd.concat([trip, trip.iloc[[0]]], ignore_index=True)
        warna = next(color_cycle)
        plt.plot(trip['Longitude'], trip['Latitude'], '-o',
                 label=f'Cluster {cluster_id} - Kendaraan {kendaraan}',
                 alpha=0.75, color=warna)
        plt.scatter(trip.iloc[0]['Longitude'], trip.iloc[0]['Latitude'],
                    c='black', marker='X', s=150)

plt.title('Rute VRP (Setiap Kendaraan Berbeda Warna)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

## **Visualisasi Folium**

In [ ]:
import folium
from folium.plugins import PolyLineTextPath

# Daftar warna yang didukung Folium
folium_vehicle_colors = [
    'red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred',
    'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white',
    'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray'
]

# Ganti 'Vehicle' ke 'Kendaraan' jika pakai kolom 'Kendaraan'
kolom_vehicle = 'Kendaraan' if 'Kendaraan' in df_rute_final.columns else 'Vehicle'

# Mapping (Cluster, Vehicle) ke warna
vehicle_ids = (
    df_rute_final[['Cluster', kolom_vehicle]]
    .drop_duplicates()
    .sort_values(['Cluster', kolom_vehicle])
    .reset_index(drop=True)
)
vehicle_ids['color'] = [folium_vehicle_colors[i % len(folium_vehicle_colors)] for i in range(len(vehicle_ids))]
vehicle_color_map = {(row['Cluster'], row[kolom_vehicle]): row['color'] for _, row in vehicle_ids.iterrows()}

# Peta awal di depot
m = folium.Map(location=dlh_coord, zoom_start=13)

for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id]
    for v in sorted(r[kolom_vehicle].unique()):
        trip = r[r[kolom_vehicle]==v].sort_values('Urutan')
        # Tutup loop ke depot jika belum
        if not (
            np.isclose(trip.iloc[0]['Longitude'], trip.iloc[-1]['Longitude']) and
            np.isclose(trip.iloc[0]['Latitude'], trip.iloc[-1]['Latitude'])
        ):
            trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        points = list(zip(trip['Latitude'], trip['Longitude']))
        color = vehicle_color_map[(cluster_id, v)]
        line = folium.PolyLine(points, color=color, weight=4, opacity=0.7).add_to(m)
        PolyLineTextPath(line, '➤   ', repeat=True, offset=6, attributes={'fill': color}).add_to(m)
        for _, row in trip.iterrows():
            folium.Marker(
                location=[row['Latitude'], row['Longitude']],
                popup=f"{row['Nama Taman']} (Cluster {cluster_id}, Vehicle {v}, Urutan {row['Urutan']})",
                icon=folium.Icon(color=color)
            ).add_to(m)

# Marker untuk depot
folium.Marker(
    location=dlh_coord,
    popup="DLH (Depot)",
    icon=folium.Icon(color="black", icon="home")
).add_to(m)

m

## **Detail Jarak Antar Taman per Cluster (Urutan Rute ACO)**

In [ ]:
from geopy.distance import geodesic
from tabulate import tabulate

kolom_vehicle = 'Kendaraan' if 'Kendaraan' in df_rute_final.columns else 'Vehicle'

for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    print(f"\n{'='*30}\nRUTE CLUSTER {cluster_id}\n{'='*30}")
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id].sort_values([kolom_vehicle, 'Urutan']).reset_index(drop=True)
    for v in r[kolom_vehicle].unique():
        trip = r[r[kolom_vehicle] == v].sort_values('Urutan').reset_index(drop=True)
        trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        output_rows = []
        for i in range(len(trip) - 1):
            a, b = trip.iloc[i], trip.iloc[i+1]
            jarak = geodesic((a['Latitude'], a['Longitude']), (b['Latitude'], b['Longitude'])).kilometers
            # Kategori fallback ke 'Status' jika tidak ada
            status_a = a['Kategori'] if 'Kategori' in a else a.get('Status', '-')
            status_b = b['Kategori'] if 'Kategori' in b else b.get('Status', '-')
            # Skip baris depot ke depot jika jaraknya 0 (atau sangat kecil, toleransi <0.01 km)
            if (a['Nama Taman'] == b['Nama Taman']) and (jarak < 0.01):
                continue
            output_rows.append({
                'Dari': f"{a['Nama Taman']} ({status_a})",
                'Ke': f"{b['Nama Taman']} ({status_b})",
                'Jarak (km)': f"{jarak:.2f}"
            })
        print(f"\n--- {kolom_vehicle} {v} ---")
        print(tabulate(output_rows, headers='keys', tablefmt='psql', showindex=False))

## **Total Jarak & Waktu Tempuh Truk per Cluster (Fixed Speed 30 km/jam)**

In [ ]:
kecepatan_truk_kmh = KECEPATAN_TRUK_KMH
hasil_per_cluster = []
for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id]
    for v in r['Kendaraan'].unique():
        trip = r[r['Kendaraan']==v].sort_values('Urutan')
        trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        total_jarak = 0
        n_taman = len(trip) - 2  # exclude DLH start & finish
        for i in range(len(trip) - 1):
            coord_a = (trip.iloc[i]['Latitude'], trip.iloc[i]['Longitude'])
            coord_b = (trip.iloc[i+1]['Latitude'], trip.iloc[i+1]['Longitude'])
            total_jarak += geodesic(coord_a, coord_b).kilometers

        waktu_jalan_jam = total_jarak / kecepatan_truk_kmh
        waktu_inspeksi_jam = n_taman * 0.5
        waktu_jam = waktu_jalan_jam + waktu_inspeksi_jam

        jam = int(waktu_jam)
        menit = int((waktu_jam - jam) * 60)
        waktu_tempuh_format = f"{jam} jam {menit} menit" if jam > 0 else f"{menit} menit"
        hasil_per_cluster.append({
            'Cluster': cluster_id,
            'Vehicle': v,
            'Jumlah Taman': n_taman,
            'Total Jarak (km)': round(total_jarak, 2),
            'Waktu Tempuh': waktu_tempuh_format
        })
hasil_df = pd.DataFrame(hasil_per_cluster)
print(tabulate(hasil_df, headers='keys', tablefmt='psql', showindex=False))

# **ACO+2OPT-VRP**

## **Distance Matrix Construction for ACO+2-Opt-VRP**

In [ ]:
from geopy.distance import geodesic
import numpy as np

def create_distance_matrix(coords):
    n = len(coords)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = geodesic(coords[i], coords[j]).kilometers
    return matrix

def total_distance(path, matrix):
    return sum(matrix[path[i]][path[i+1]] for i in range(len(path)-1)) + matrix[path[-1]][path[0]]

def two_opt(route, matrix):
    best = route.copy()
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):
            for j in range(i + 1, len(route)):
                if j - i == 1: continue
                new_route = best[:i] + best[i:j][::-1] + best[j:]
                if total_distance(new_route, matrix) < total_distance(best, matrix):
                    best = new_route
                    improved = True
        route = best
    return best

## **Implementasi Fungsi ACO+2OPT untuk Vehicle Routing Problem**

In [ ]:
import random

class ACO:
    def __init__(self, n_ants, n_iterations, alpha, beta, evaporation_rate, Q):
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.beta = beta
        self.evaporation_rate = evaporation_rate
        self.Q = Q

    def _select_next_node(self, current, unvisited, pheromone, heuristic):
        probs = [pheromone[current][j]**self.alpha * heuristic[current][j]**self.beta for j in unvisited]
        probs = np.array(probs) / sum(probs)
        return random.choices(list(unvisited), weights=probs)[0]

    def solve(self, dist_matrix, start_node=0):
        n = len(dist_matrix)
        pheromone = np.ones((n, n))
        heuristic = 1 / (dist_matrix + 1e-10)
        best_path, best_dist = None, float('inf')

        for _ in range(self.n_iterations):
            all_paths = []
            for _ in range(self.n_ants):
                unvisited = set(range(n))
                path = [start_node]
                unvisited.remove(start_node)
                curr = start_node

                while unvisited:
                    nxt = self._select_next_node(curr, unvisited, pheromone, heuristic)
                    path.append(nxt)
                    unvisited.remove(nxt)
                    curr = nxt

                improved = two_opt(path, dist_matrix)
                dist = total_distance(improved, dist_matrix)
                all_paths.append((improved, dist))

                if dist < best_dist:
                    best_path = improved
                    best_dist = dist

            pheromone *= (1 - self.evaporation_rate)
            for path, dist in all_paths:
                for i in range(len(path) - 1):
                    pheromone[path[i]][path[i+1]] += self.Q / dist
                pheromone[path[-1]][path[0]] += self.Q / dist

        return best_path

In [ ]:
def two_opt(route, distance_matrix):
    best = route.copy()
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):
            for j in range(i + 1, len(route)):
                if j - i == 1: continue  # Skip neighbors
                new_route = best[:i] + best[i:j][::-1] + best[j:]
                if total_distance(new_route, distance_matrix) < total_distance(best, distance_matrix):
                    best = new_route
                    improved = True
        route = best
    return best

## **Proses VRP Setiap Cluster (DLH disisipkan di awal)**

In [ ]:
from datetime import timedelta, datetime
import numpy as np

aco_solver = ACO(n_ants=5, n_iterations=20, alpha=1, beta=5, evaporation_rate=0.5, Q=100)
cluster_results = []

kecepatan_truk_kmh = 30
maks_jam_kerja = 8
start_time_str = "08:00"

# Mapping cluster_id ke jumlah vehicle
vehicle_per_cluster = {
    0: 2,
    1: 3,
    2: 1
}

def get_dlh_df(cluster_id):
    return pd.DataFrame({
        'Nama Taman': [dlh_row['Nama Taman']],
        'Latitude': [dlh_row['Latitude']],
        'Longitude': [dlh_row['Longitude']],
        'Cluster': [cluster_id]
    })

for cluster_id in sorted(df['Cluster'].unique()):
    n_vehicle = vehicle_per_cluster.get(cluster_id, 1)
    cluster_df = df[(df['Cluster'] == cluster_id) & (df['Nama Taman'] != dlh_row['Nama Taman'])].reset_index(drop=True)
    dlh_df = get_dlh_df(cluster_id)

    # Sort taman spatially by angle from depot (bisa juga random, atau via jarak/greedy)
    depot_coord = (dlh_df.iloc[0]['Latitude'], dlh_df.iloc[0]['Longitude'])
    angle = np.arctan2(cluster_df['Latitude'] - depot_coord[0], cluster_df['Longitude'] - depot_coord[1])
    cluster_df['angle'] = angle
    cluster_df = cluster_df.sort_values('angle').reset_index(drop=True)

    # Bagi taman secara merata ke n_vehicle batch
    batch_size = int(np.ceil(len(cluster_df) / n_vehicle))
    time_start = datetime.strptime(start_time_str, "%H:%M")
    vehicle_num = 1

    for i in range(n_vehicle):
        batch_indices = list(range(i * batch_size, min((i + 1) * batch_size, len(cluster_df))))
        sub_df = pd.concat([dlh_df, cluster_df.iloc[batch_indices]], ignore_index=True)
        coords = sub_df[['Latitude', 'Longitude']].values
        dist_matrix = create_distance_matrix(coords)
        tsp_path = aco_solver.solve(dist_matrix, start_node=0)

        # Hitung waktu trip
        time_accum = 0
        last_idx = 0
        for j in tsp_path[1:]:
            travel_time = dist_matrix[last_idx][j] / kecepatan_truk_kmh
            if j == 0:
                waktu_inspeksi = 0
            else:
                kategori = sub_df.iloc[j]['Kategori'].strip().lower()
                waktu_inspeksi = 0.5 if kategori == 'pemerintah' else 0.25
            time_accum += travel_time + waktu_inspeksi
            last_idx = j
        time_accum += dist_matrix[last_idx][0] / kecepatan_truk_kmh

        # Simpan trip
        route_result = sub_df.iloc[tsp_path].copy().reset_index(drop=True)
        route_result['Urutan'] = range(1, len(route_result) + 1)
        route_result['Kendaraan'] = vehicle_num
        route_result['Cluster'] = cluster_id
        route_result['Waktu Mulai'] = time_start.strftime('%H:%M')
        waktu_selesai = (time_start + timedelta(hours=time_accum)).strftime('%H:%M')
        route_result['Waktu Selesai'] = waktu_selesai
        route_result['Total Jam'] = round(time_accum, 2)
        cluster_results.append(route_result)

        # Optional: warning jika trip over
        if time_accum > maks_jam_kerja:
            print(f"Warning: Cluster {cluster_id} Vehicle {vehicle_num} trip waktu {time_accum:.2f} jam > 8 jam!")

        time_start += timedelta(hours=time_accum)
        vehicle_num += 1

df_rute_final = pd.concat(cluster_results, ignore_index=True)

## **Visualisasi Pembagian Cluster**

In [ ]:
plt.figure(figsize=(10,8))
colors = ['green', 'red', 'blue']
for cluster_id, color in zip(df['Cluster'].unique(), colors):
    cluster_df2 = df[df['Cluster'] == cluster_id]
    plt.scatter(cluster_df2['Longitude'], cluster_df2['Latitude'], label=f'Cluster {cluster_id}', color=color, s=60)
plt.title('Pembagian Cluster Taman (Hasil KMeans)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

## **Visualisasi Rute VRP Optimal + DLH**

In [ ]:
plt.figure(figsize=(10,8))
for (cluster_id, color) in zip(sorted(df_rute_final['Cluster'].unique()), colors):
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id].sort_values(['Kendaraan', 'Urutan'])
    for v in r['Kendaraan'].unique():
        trip = r[r['Kendaraan']==v].sort_values('Urutan')
        trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        plt.plot(trip['Longitude'], trip['Latitude'], '-o', label=f'Cluster {cluster_id} Vehicle {v}', alpha=0.7)
plt.scatter(dlh_coord[1], dlh_coord[0], c='black', marker='X', s=200, label='DLH (Depot)')
plt.title('Rute VRP Multi-Trip (ACO + 2OPT, Feasible Time)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

## **Visualisasi Folium**

In [ ]:
# Daftar warna folium yang didukung
folium_vehicle_colors = [
    'red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred',
    'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white',
    'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray'
]

# Buat mapping: (Cluster, Vehicle) ke warna
vehicle_ids = df_rute_final[['Cluster', 'Kendaraan']].drop_duplicates().sort_values(['Cluster', 'Kendaraan']).reset_index(drop=True)
vehicle_ids['color'] = [folium_vehicle_colors[i % len(folium_vehicle_colors)] for i in range(len(vehicle_ids))]
vehicle_color_map = {(row['Cluster'], row['Kendaraan']): row['color'] for _, row in vehicle_ids.iterrows()}

m = folium.Map(location=dlh_coord, zoom_start=13)

for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id]
    for v in r['Kendaraan'].unique():
        trip = r[r['Kendaraan']==v].sort_values('Urutan')
        trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        points = list(zip(trip['Latitude'], trip['Longitude']))
        color = vehicle_color_map[(cluster_id, v)]
        line = folium.PolyLine(points, color=color, weight=4, opacity=0.7).add_to(m)
        PolyLineTextPath(line, '➤   ', repeat=True, offset=6, attributes={'fill': color}).add_to(m)
        for _, row in trip.iterrows():
            folium.Marker(
                location=[row['Latitude'], row['Longitude']],
                popup=f"{row['Nama Taman']} (Cluster {cluster_id}, Vehicle {v}, Urutan {row['Urutan']})",
                icon=folium.Icon(color=color)
            ).add_to(m)
folium.Marker(location=dlh_coord, popup="DLH (Depot)", icon=folium.Icon(color="black", icon="home")).add_to(m)
m

## **Detail Jarak Antar Taman per Cluster (Urutan Rute ACO)**

In [ ]:
for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    print(f"\n{'='*30}\nRUTE CLUSTER {cluster_id}\n{'='*30}")
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id].sort_values(['Kendaraan', 'Urutan']).reset_index(drop=True)
    for v in r['Kendaraan'].unique():
        trip = r[r['Kendaraan'] == v].sort_values('Urutan').reset_index(drop=True)
        trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        output_rows = []
        for i in range(len(trip) - 1):
            a, b = trip.iloc[i], trip.iloc[i+1]
            jarak = geodesic((a['Latitude'], a['Longitude']), (b['Latitude'], b['Longitude'])).kilometers
            # Kolom status: pilih nama kolom sesuai dataframe-mu (Kategori atau Status)
            status_a = a['Kategori'] if 'Kategori' in a else a['Status']
            status_b = b['Kategori'] if 'Kategori' in b else b['Status']
            output_rows.append({
                'Dari': f"{a['Nama Taman']} ({status_a})",
                'Ke': f"{b['Nama Taman']} ({status_b})",
                'Jarak (km)': f"{jarak:.2f}"
            })
        print(f"\n--- Kendaraan {v} ---")
        print(tabulate(output_rows, headers='keys', tablefmt='psql', showindex=False))

## **Total Jarak & Waktu Tempuh Truk per Cluster (Fixed Speed 30 km/jam)**

In [ ]:
hasil_per_cluster = []
for cluster_id in sorted(df_rute_final['Cluster'].unique()):
    r = df_rute_final[df_rute_final['Cluster'] == cluster_id]
    for v in r['Kendaraan'].unique():
        trip = r[r['Kendaraan']==v].sort_values('Urutan')
        trip = pd.concat([trip, trip.iloc[0:1]], ignore_index=True)
        total_jarak = 0
        n_taman = len(trip) - 2  # exclude DLH start & finish
        for i in range(len(trip) - 1):
            coord_a = (trip.iloc[i]['Latitude'], trip.iloc[i]['Longitude'])
            coord_b = (trip.iloc[i+1]['Latitude'], trip.iloc[i+1]['Longitude'])
            total_jarak += geodesic(coord_a, coord_b).kilometers

        waktu_jalan_jam = total_jarak / kecepatan_truk_kmh
        waktu_inspeksi_jam = n_taman * 0.5
        waktu_jam = waktu_jalan_jam + waktu_inspeksi_jam

        jam = int(waktu_jam)
        menit = int((waktu_jam - jam) * 60)
        waktu_tempuh_format = f"{jam} jam {menit} menit" if jam > 0 else f"{menit} menit"
        hasil_per_cluster.append({
            'Cluster': cluster_id,
            'Vehicle': v,
            'Jumlah Taman': n_taman,
            'Total Jarak (km)': round(total_jarak, 2),
            'Waktu Tempuh': waktu_tempuh_format
        })
hasil_df = pd.DataFrame(hasil_per_cluster)
print(tabulate(hasil_df, headers='keys', tablefmt='psql', showindex=False))